In [1]:
import $ivy.`org.apache.spark::spark-sql:3.5.8`
import $ivy.`org.apache.spark::spark-core:3.5.8`

import $ivy.$
import $ivy.$

In [2]:
import org.apache.spark.sql._
import org.apache.spark.sql.types._
import org.apache.spark.sql.functions._

val spark = { NotebookSparkSession
                .builder()
                .master("local[*]")
                .appName("NYTaxi")
                .config("spark.log.level", "WARN")
                .config("spark.driver.memory", "16g")
                .getOrCreate() }

import spark.implicits._

Loading spark-stubs

Creating SparkSession

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/04 18:12:21 INFO SparkContext: Running Spark version 3.5.8
26/08/04 18:12:21 INFO SparkContext: OS info Linux, 6.12.0-211.34.1.el10_2.x86_64, amd64
26/08/04 18:12:21 INFO SparkContext: Java version 11.0.31
26/08/04 18:12:22 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting Spark log level to "WARN".


Spark UI

import org.apache.spark.sql._
import org.apache.spark.sql.types._
import org.apache.spark.sql.functions._
spark: SparkSession = org.apache.spark.sql.SparkSession@64bf4514
import spark.implicits._

# Загрузка и очистка данных

Нью-Йоркская комиссия по такси и лимузинам (TLC) предоставляет подробные анонимизированные данные о поездках клиентов с 2009 года. Машины, выкрашенные жёлтым, имеют право брать пассажиров на улицах в любом из пяти боро города.

## Загружаем данные о поездках

In [3]:

val tripSchema = StructType(
                  Array(
                    StructField("VendorID", IntegerType, true),
                    StructField("tpep_pickup_datetime", TimestampType, true),
                    StructField("tpep_dropoff_datetime", TimestampType, true),
                    StructField("passenger_count", IntegerType, true),
                    StructField("trip_distance", DoubleType, true),
                    StructField("pickup_longitude", DoubleType, true),
                    StructField("pickup_latitude", DoubleType, true),
                    StructField("RatecodeID", IntegerType, true),
                    StructField("store_and_fwd_flag", StringType, true),
                    StructField("dropoff_longitude", DoubleType, true),
                    StructField("dropoff_latitude", DoubleType, true),
                    StructField("payment_type", IntegerType, true),
                    StructField("fare_amount", DoubleType, true),
                    StructField("extra", DoubleType, true),
                    StructField("mta_tax", DoubleType, true),
                    StructField("tip_amount", DoubleType, true),
                    StructField("tolls_amount", DoubleType, true),
                    StructField("improvement_surcharge", DoubleType, true),
                    StructField("total_amount", DoubleType, true)))

tripSchema: StructType = StructType(
  StructField("VendorID", IntegerType, true, {}),
  StructField("tpep_pickup_datetime", TimestampType, true, {}),
  StructField("tpep_dropoff_datetime", TimestampType, true, {}),
  StructField("passenger_count", IntegerType, true, {}),
  StructField("trip_distance", DoubleType, true, {}),
  StructField("pickup_longitude", DoubleType, true, {}),
  StructField("pickup_latitude", DoubleType, true, {}),
  StructField("RatecodeID", IntegerType, true, {}),
  StructField("store_and_fwd_flag", StringType, true, {}),
  StructField("dropoff_longitude", DoubleType, true, {}),
  StructField("dropoff_latitude", DoubleType, true, {}),
  StructField("payment_type", IntegerType, true, {}),
  StructField("fare_amount", DoubleType, true, {}),
  StructField("extra", DoubleType, true, {}),
  StructField("mta_tax", DoubleType, true, {}),
  StructField("tip_amount", DoubleType, true, {}),
  StructField("tolls_amount", DoubleType, true, {}),
  StructField("improvement_sur

In [4]:
val raw = spark
            .read
            .format("csv")
            .option("header", "true")
            .schema(tripSchema)
            .load("data/trips/2016")

println(s"\nКоличество записей: ${raw.count}")


Количество записей: 34499859


raw: DataFrame = [VendorID: int, tpep_pickup_datetime: timestamp ... 17 more fields]

## Проверка и очистка данных

In [5]:
raw.show(5, false)

+--------+--------------------+---------------------+---------------+-------------+------------------+------------------+----------+------------------+------------------+------------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|pickup_longitude  |pickup_latitude   |RatecodeID|store_and_fwd_flag|dropoff_longitude |dropoff_latitude  |payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|
+--------+--------------------+---------------------+---------------+-------------+------------------+------------------+----------+------------------+------------------+------------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+
|2       |2016-02-25 17:24:20 |2016-02-25 17:27:20  |2              |0.7          |-73.94725036621094|40.763771057128906|1         |N   

### Фильтрация данных

Удалим записи о поездках с:
- нулевой длительностью
- нулевым количеством пассажиров
- нулевым расстоянием поездки по счётчику
- координатами начала, не попадающими в прямоугольник Нью-Йорка

Нью-Йорк вписан в прямоугольник от -74.25559 до -73.70001 градусов долготы и от 40.49612 до 40.91553 широты.

In [7]:
implicitly[almond.api.JupyterApi]
import almond.display._

Html("""<!DOCTYPE html>
<html xmlns="http://www.w3.org/1999/xhtml">
<head>
    <meta http-equiv="Content-Type" content="text/html; charset=utf-8" />
    <script src="https://api-maps.yandex.ru/2.1/?lang=ru_RU" type="text/javascript"></script>
    <script type="text/javascript">
        ymaps.ready(init);
        var myMap;
        function init(){ 
            myMap = new ymaps.Map("map", {
                center: [40.705825, -73.9778],
                zoom: 11
            }); 
            var nyRectangle = new ymaps.Rectangle([[40.49612, -74.25559],[40.91553, -73.70001]], {}, {fillColor: "00666666"});
            myMap.geoObjects.add(nyRectangle);
        }
    </script>
</head>
<body>
    <div id="map" style="width: 1000px; height: 1000px"></div>
</body>
</html>"""
)

<!DOCTYPE html>

res7_0: almond.api.JupyterApi = almond.JupyterApiImpl@61e8b0ab
import almond.display._

In [8]:
val ny_min_lon = -74.25559
val ny_max_lon = -73.70001
val ny_min_lat = 40.49612
val ny_max_lat = 40.91553

ny_min_lon: Double = -74.25559
ny_max_lon: Double = -73.70001
ny_min_lat: Double = 40.49612
ny_max_lat: Double = 40.91553

In [9]:
val trips = raw
              .filter($"tpep_pickup_datetime" < $"tpep_dropoff_datetime")
              .filter($"passenger_count" > 0)
              .filter($"trip_distance" > 0)
              .filter($"pickup_longitude" >= ny_min_lon)
              .filter($"pickup_longitude" <= ny_max_lon)
              .filter($"pickup_latitude" >= ny_min_lat)
              .filter($"pickup_latitude" <= ny_max_lat)
              .filter($"tpep_pickup_datetime" >= "2016-01-01 00:00:00")
              .filter($"tpep_pickup_datetime" <= "2016-03-31 23:59:59")

trips: Dataset[Row] = [VendorID: int, tpep_pickup_datetime: timestamp ... 17 more fields]

In [10]:
trips.show(5, false)

+--------+--------------------+---------------------+---------------+-------------+------------------+------------------+----------+------------------+------------------+------------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|pickup_longitude  |pickup_latitude   |RatecodeID|store_and_fwd_flag|dropoff_longitude |dropoff_latitude  |payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|
+--------+--------------------+---------------------+---------------+-------------+------------------+------------------+----------+------------------+------------------+------------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+
|2       |2016-02-25 17:24:20 |2016-02-25 17:27:20  |2              |0.7          |-73.94725036621094|40.763771057128906|1         |N   

## Геопространственный анализ

### Подключаем Apache Sedona

In [11]:
import $ivy.`org.apache.sedona:sedona-spark-shaded-3.5_2.12:1.9.0`
import $ivy.`org.datasyslab:geotools-wrapper:1.9.0-33.5`

import $ivy.$
import $ivy.$

In [12]:
import org.apache.sedona.spark.SedonaContext

val config = SedonaContext
                .builder()
                .master("local[*]")
                .appName("NYTaxi")
                .getOrCreate()

val sedona = SedonaContext.create(config)

26/08/04 18:13:19 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


import org.apache.sedona.spark.SedonaContext
config: SparkSession = org.apache.spark.sql.SparkSession@64bf4514
sedona: SparkSession = org.apache.spark.sql.SparkSession@64bf4514

### Преобразуем координаты в точки

In [13]:
import org.apache.spark.sql.sedona_sql.expressions.st_functions._
import org.apache.spark.sql.sedona_sql.expressions.st_constructors._
import org.apache.spark.sql.sedona_sql.expressions.st_predicates._
import org.apache.spark.sql.sedona_sql.expressions.st_aggregates._

import org.apache.spark.sql.sedona_sql.expressions.st_functions._
import org.apache.spark.sql.sedona_sql.expressions.st_constructors._
import org.apache.spark.sql.sedona_sql.expressions.st_predicates._
import org.apache.spark.sql.sedona_sql.expressions.st_aggregates._

In [14]:
val tripsPoints = trips
                    .withColumn("pickupPoint", ST_Point($"pickup_longitude", $"pickup_latitude"))
                    .withColumn("dropoffPoint", ST_Point($"dropoff_longitude", $"dropoff_latitude"))

tripsPoints: DataFrame = [VendorID: int, tpep_pickup_datetime: timestamp ... 19 more fields]

In [15]:
tripsPoints.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- pickup_longitude: double (nullable = true)
 |-- pickup_latitude: double (nullable = true)
 |-- RatecodeID: integer (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- dropoff_longitude: double (nullable = true)
 |-- dropoff_latitude: double (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- pickupPoint: geometry (nullable = true)
 |-- dropoffPoint: geometry (nullable = true)



In [16]:
tripsPoints.show(5)

+--------+--------------------+---------------------+---------------+-------------+------------------+------------------+----------+------------------+------------------+------------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+--------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|  pickup_longitude|   pickup_latitude|RatecodeID|store_and_fwd_flag| dropoff_longitude|  dropoff_latitude|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|         pickupPoint|        dropoffPoint|
+--------+--------------------+---------------------+---------------+-------------+------------------+------------------+----------+------------------+------------------+------------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+--------------------+
|       2|

In [17]:
val tripsData = tripsPoints
                  .select(
                    $"tpep_pickup_datetime".as("pickupDatetime"),
                    $"pickupPoint",
                    $"passenger_count",
                    $"trip_distance",
                    $"total_amount",
                    $"tpep_dropoff_datetime".as("dropoffDatetime"),
                    $"dropoffPoint")

tripsData: DataFrame = [pickupDatetime: timestamp, pickupPoint: geometry ... 5 more fields]

In [18]:
tripsData.show(5)

+-------------------+--------------------+---------------+-------------+------------+-------------------+--------------------+
|     pickupDatetime|         pickupPoint|passenger_count|trip_distance|total_amount|    dropoffDatetime|        dropoffPoint|
+-------------------+--------------------+---------------+-------------+------------+-------------------+--------------------+
|2016-02-25 17:24:20|POINT (-73.947250...|              2|          0.7|         5.8|2016-02-25 17:27:20|POINT (-73.992012...|
|2016-02-25 23:10:50|POINT (-73.983016...|              2|         5.52|        21.3|2016-02-25 23:31:50|POINT (-73.988586...|
|2016-02-01 00:00:01|POINT (-73.992340...|              6|         1.99|        11.5|2016-02-01 00:10:52|POINT (-73.964355...|
|2016-02-01 00:00:04|POINT (-73.981452...|              1|          1.5|         7.8|2016-02-01 00:05:16|POINT (-73.982322...|
|2016-02-01 00:00:05|POINT (-74.000602...|              1|          5.6|        25.3|2016-02-01 00:20:59|POINT 

### Загружаем данные о зонах в GeoJSON

#### Вариант 1

In [19]:
val geoZones = sedona
                 .read
                 .format("geojson")
                 .option("multiLine", "true")
                 .load("data/NYC_Taxi_Zones.geojson")
                 .selectExpr("explode(features) as features")
                 .select("features.*")
                 .withColumn("objectID", expr("properties['objectid']").cast(IntegerType))
                 .withColumn("locationID", expr("properties['location_id']").cast(IntegerType))
                 .withColumn("zone", expr("properties['zone']"))
                 .withColumn("borough", expr("properties['borough']"))
                 .withColumn("shapeArea", expr("properties['shape_area']").cast(DoubleType))
                 .withColumn("shapeLeng", expr("properties['shape_leng']").cast(DoubleType))
                 .drop("properties", "type")

geoZones: DataFrame = [geometry: geometry, objectID: int ... 5 more fields]

In [20]:
geoZones.printSchema()

root
 |-- geometry: geometry (nullable = true)
 |-- objectID: integer (nullable = true)
 |-- locationID: integer (nullable = true)
 |-- zone: string (nullable = true)
 |-- borough: string (nullable = true)
 |-- shapeArea: double (nullable = true)
 |-- shapeLeng: double (nullable = true)



In [21]:
geoZones.show(5)

+--------------------+--------+----------+--------------------+-------------+----------------+---------------+
|            geometry|objectID|locationID|                zone|      borough|       shapeArea|      shapeLeng|
+--------------------+--------+----------+--------------------+-------------+----------------+---------------+
|MULTIPOLYGON (((-...|       1|         1|      Newark Airport|          EWR|  7.823067885E-4| 0.116357453189|
|MULTIPOLYGON (((-...|       2|         2|         Jamaica Bay|       Queens|0.00486634037837|  0.43346966679|
|MULTIPOLYGON (((-...|       3|         3|Allerton/Pelham G...|        Bronx|3.14414156821E-4|0.0843411059012|
|MULTIPOLYGON (((-...|       4|         4|       Alphabet City|    Manhattan|1.11871946192E-4|0.0435665270921|
|MULTIPOLYGON (((-...|       5|         5|       Arden Heights|Staten Island|4.97957489363E-4|0.0921464898574|
+--------------------+--------+----------+--------------------+-------------+----------------+---------------+
o

Оставляем названия зоны, боро и геометрию.

In [22]:
val zones = geoZones.select("zone", "borough", "geometry").cache()

zones: Dataset[Row] = [zone: string, borough: string ... 1 more field]

In [23]:
zones.show(5)

+--------------------+-------------+--------------------+
|                zone|      borough|            geometry|
+--------------------+-------------+--------------------+
|      Newark Airport|          EWR|MULTIPOLYGON (((-...|
|         Jamaica Bay|       Queens|MULTIPOLYGON (((-...|
|Allerton/Pelham G...|        Bronx|MULTIPOLYGON (((-...|
|       Alphabet City|    Manhattan|MULTIPOLYGON (((-...|
|       Arden Heights|Staten Island|MULTIPOLYGON (((-...|
+--------------------+-------------+--------------------+
only showing top 5 rows



#### Вариант 2

In [24]:
val shapes = sedona.read.format("shapefile").load("data/zones")

shapes: DataFrame = [geometry: geometry, OBJECTID: bigint ... 5 more fields]

In [25]:
shapes.printSchema()

root
 |-- geometry: geometry (nullable = true)
 |-- OBJECTID: long (nullable = true)
 |-- Shape_Leng: decimal(19,11) (nullable = true)
 |-- Shape_Area: decimal(19,11) (nullable = true)
 |-- zone: string (nullable = true)
 |-- LocationID: long (nullable = true)
 |-- borough: string (nullable = true)



In [26]:
shapes.show(5)

+--------------------+--------+-------------+-------------+--------------------+----------+-------------+
|            geometry|OBJECTID|   Shape_Leng|   Shape_Area|                zone|LocationID|      borough|
+--------------------+--------+-------------+-------------+--------------------+----------+-------------+
|POLYGON ((933100....|       1|0.11635745319|0.00078230679|      Newark Airport|         1|          EWR|
|MULTIPOLYGON (((1...|       2|0.43346966679|0.00486634038|         Jamaica Bay|         2|       Queens|
|POLYGON ((1026308...|       3|0.08434110590|0.00031441416|Allerton/Pelham G...|         3|        Bronx|
|POLYGON ((992073....|       4|0.04356652709|0.00011187195|       Alphabet City|         4|    Manhattan|
|POLYGON ((935843....|       5|0.09214648986|0.00049795749|       Arden Heights|         5|Staten Island|
+--------------------+--------+-------------+-------------+--------------------+----------+-------------+
only showing top 5 rows



In [28]:
val zones2 = shapes.select($"zone", $"borough", $"geometry")

zones2: DataFrame = [zone: string, borough: string ... 1 more field]

In [29]:
zones2.show(5)

+--------------------+-------------+--------------------+
|                zone|      borough|            geometry|
+--------------------+-------------+--------------------+
|      Newark Airport|          EWR|POLYGON ((933100....|
|         Jamaica Bay|       Queens|MULTIPOLYGON (((1...|
|Allerton/Pelham G...|        Bronx|POLYGON ((1026308...|
|       Alphabet City|    Manhattan|POLYGON ((992073....|
|       Arden Heights|Staten Island|POLYGON ((935843....|
+--------------------+-------------+--------------------+
only showing top 5 rows



### Соединяем поездки с зонами

#### Вариант 1

In [30]:
val tripsZones = tripsData
                    .join(
                        broadcast(zones),
                        ST_Contains(zones("geometry"), tripsData("pickupPoint")),
                        "left")
                    .withColumnRenamed("zone", "pickupZone")
                    .withColumnRenamed("borough", "pickupBorough")
                    .drop("geometry")
                    .join(
                        broadcast(zones),
                        ST_Contains(zones("geometry"), tripsData("dropoffPoint")),
                        "left")
                    .withColumnRenamed("zone", "dropoffZone")
                    .withColumnRenamed("borough", "dropoffBorough")
                    .drop("geometry")
                    .na.fill("NA", Seq("pickupZone", "pickupBorough", "dropoffZone", "dropoffBorough"))
                    .cache()

tripsZones: Dataset[Row] = [pickupDatetime: timestamp, pickupPoint: geometry ... 9 more fields]

In [31]:
tripsZones.printSchema()

root
 |-- pickupDatetime: timestamp (nullable = true)
 |-- pickupPoint: geometry (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- dropoffDatetime: timestamp (nullable = true)
 |-- dropoffPoint: geometry (nullable = true)
 |-- pickupZone: string (nullable = false)
 |-- pickupBorough: string (nullable = false)
 |-- dropoffZone: string (nullable = false)
 |-- dropoffBorough: string (nullable = false)



In [32]:
tripsZones.show(5)

+-------------------+--------------------+---------------+-------------+------------+-------------------+--------------------+--------------------+-------------+--------------------+--------------+
|     pickupDatetime|         pickupPoint|passenger_count|trip_distance|total_amount|    dropoffDatetime|        dropoffPoint|          pickupZone|pickupBorough|         dropoffZone|dropoffBorough|
+-------------------+--------------------+---------------+-------------+------------+-------------------+--------------------+--------------------+-------------+--------------------+--------------+
|2016-02-25 17:24:20|POINT (-73.947250...|              2|          0.7|         5.8|2016-02-25 17:27:20|POINT (-73.992012...|    Roosevelt Island|    Manhattan|            Union Sq|     Manhattan|
|2016-02-25 23:10:50|POINT (-73.983016...|              2|         5.52|        21.3|2016-02-25 23:31:50|POINT (-73.988586...|       Midtown South|    Manhattan|Times Sq/Theatre ...|     Manhattan|
|2016-02-0

#### Вариант 2

In [33]:
import org.locationtech.jts.geom._
import org.apache.sedona.common._

case class Zones(zone: String, borough: String, geometry: Geometry)

import org.locationtech.jts.geom._
import org.apache.sedona.common._
defined class Zones

In [34]:
val zonesLocal = zones.as[Zones].collect().map(z => (z.zone, z.borough, z.geometry))
val zonesLocalBC = spark.sparkContext.broadcast(zonesLocal)

zonesLocal: Array[(String, String, Geometry)] = Array(
  (
    "Newark Airport",
    "EWR",
    MULTIPOLYGON (((-74.18445299999996 40.694995999999904, -74.18448899999999 40.69509499999987, -74.18449799999996 40.69518499999987, -74.18438099999997 40.69587799999989, -74.18428199999994 40.6962109999999, -74.18402099999997 40.697074999999884, -74.18391299999996 40.69750699999986, -74.18375099999997 40.69779499999988, -74.18363399999998 40.6983259999999, -74.18356199999994 40.698451999999875, -74.18354399999998 40.69855999999988, -74.18350799999996 40.69870399999992, -74.18327399999998 40.70008999999988, -74.18315699999994 40.701214999999884, -74.18316599999997 40.702384999999886, -74.18313899999998 40.7026279999999, -74.18309399999998 40.7028529999999, -74.18299499999995 40.70315899999985, -74.18284199999994 40.70346499999989, -74.18264399999998 40.70373499999988, -74.18242799999996 40.70395099999992, -74.18220299999996 40.704139999999896, -74.18203199999994 40.70425699999987, -74.18180699

In [35]:
val pointInZone = udf((point: Geometry) => {
    val f = zonesLocalBC.value.filter(z => Predicates.contains(z._3, point))
    if (f.isEmpty) Array("NA", "NA")
    else Array(f(0)._1, f(0)._2)
})

pointInZone: expressions.UserDefinedFunction = SparkUserDefinedFunction(
  ammonite.$sess.cmd35$Helper$$Lambda$8532/0x0000000841d0e840@15dd20cc,
  ArrayType(StringType, true),
  List(
    Some(
      ExpressionEncoder(
        Invoke(
          NewInstance(
            class org.apache.spark.sql.sedona_sql.UDT.GeometryUDT,
            List(),
            List(),
            true,
            ObjectType(class org.apache.spark.sql.sedona_sql.UDT.GeometryUDT),
            None
          ),
          "serialize",
          GeometryUDT,
          List(
            BoundReference(
              0,
              ObjectType(class org.locationtech.jts.geom.Geometry),
              true
            )
          ),
          List(),
          true,
          true,
          true
        ),
        Invoke(
          NewInstance(
            class org.apache.spark.sql.sedona_sql.UDT.GeometryUDT,
            List(),
            List(),
            true,
            ObjectType(class org.apache.spark.s

In [36]:
val tripsZones2 = tripsData
                    .withColumn("pickupZB", pointInZone($"pickupPoint"))
                    .withColumn("pickupZone", $"pickupZB"(0))
                    .withColumn("pickupBorough", $"pickupZB"(1))
                    .drop("pickupZB")
                    .withColumn("dropoffZB", pointInZone($"dropoffPoint"))
                    .withColumn("dropoffZone", $"dropoffZB"(0))
                    .withColumn("dropoffBorough", $"dropoffZB"(1))
                    .drop("dropoffZB")

tripsZones2: DataFrame = [pickupDatetime: timestamp, pickupPoint: geometry ... 9 more fields]

In [37]:
tripsZones2.printSchema()

root
 |-- pickupDatetime: timestamp (nullable = true)
 |-- pickupPoint: geometry (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- dropoffDatetime: timestamp (nullable = true)
 |-- dropoffPoint: geometry (nullable = true)
 |-- pickupZone: string (nullable = true)
 |-- pickupBorough: string (nullable = true)
 |-- dropoffZone: string (nullable = true)
 |-- dropoffBorough: string (nullable = true)



In [38]:
tripsZones2.show(5)

26/08/04 18:18:28 WARN DAGScheduler: Broadcasting large task binary with size 2.9 MiB


+-------------------+--------------------+---------------+-------------+------------+-------------------+--------------------+--------------------+-------------+--------------------+--------------+
|     pickupDatetime|         pickupPoint|passenger_count|trip_distance|total_amount|    dropoffDatetime|        dropoffPoint|          pickupZone|pickupBorough|         dropoffZone|dropoffBorough|
+-------------------+--------------------+---------------+-------------+------------+-------------------+--------------------+--------------------+-------------+--------------------+--------------+
|2016-02-25 17:24:20|POINT (-73.947250...|              2|          0.7|         5.8|2016-02-25 17:27:20|POINT (-73.992012...|    Roosevelt Island|    Manhattan|            Union Sq|     Manhattan|
|2016-02-25 23:10:50|POINT (-73.983016...|              2|         5.52|        21.3|2016-02-25 23:31:50|POINT (-73.988586...|       Midtown South|    Manhattan|Times Sq/Theatre ...|     Manhattan|
|2016-02-0

In [39]:
val rides = tripsZones2.select(
                          "pickupDatetime",
                          "pickupZone",
                          "pickupBorough",
                          "dropoffDatetime",
                          "dropoffZone",
                          "dropoffBorough",
                          "passenger_count",
                          "trip_distance",
                          "total_amount")

rides: DataFrame = [pickupDatetime: timestamp, pickupZone: string ... 7 more fields]

### Агрегация данных по часам и районам посадки

In [43]:
val tripsMDH = rides
              .withColumn("month", month($"pickupDatetime"))
              .withColumn("day", dayofmonth($"pickupDatetime"))
              .withColumn("hour", hour($"pickupDatetime"))

tripsMDH: DataFrame = [pickupDatetime: timestamp, pickupZone: string ... 10 more fields]

In [44]:
tripsMDH.show(5)

26/08/04 18:33:43 WARN DAGScheduler: Broadcasting large task binary with size 2.9 MiB


+-------------------+--------------------+-------------+-------------------+--------------------+--------------+---------------+-------------+------------+-----+---+----+
|     pickupDatetime|          pickupZone|pickupBorough|    dropoffDatetime|         dropoffZone|dropoffBorough|passenger_count|trip_distance|total_amount|month|day|hour|
+-------------------+--------------------+-------------+-------------------+--------------------+--------------+---------------+-------------+------------+-----+---+----+
|2016-02-25 17:24:20|    Roosevelt Island|    Manhattan|2016-02-25 17:27:20|            Union Sq|     Manhattan|              2|          0.7|         5.8|    2| 25|  17|
|2016-02-25 23:10:50|       Midtown South|    Manhattan|2016-02-25 23:31:50|Times Sq/Theatre ...|     Manhattan|              2|         5.52|        21.3|    2| 25|  23|
|2016-02-01 00:00:01|        Clinton East|    Manhattan|2016-02-01 00:10:52|Sutton Place/Turt...|     Manhattan|              6|         1.99|   

In [45]:
val tripsAgg = tripsMDH
                .groupBy($"month", $"day", $"hour", $"pickupZone")
                .agg(
                    count("*").as("trips"),
                    avg("trip_distance").as("avg_distance"),
                    avg("total_amount").as("avg_amount"))
                .orderBy($"month", $"day", $"hour", $"pickupZone")
                .cache()

tripsAgg: Dataset[Row] = [month: int, day: int ... 5 more fields]

In [46]:
tripsAgg.show(5)

26/08/04 18:35:51 WARN DAGScheduler: Broadcasting large task binary with size 3.0 MiB


26/08/04 18:41:21 WARN DAGScheduler: Broadcasting large task binary with size 3.0 MiB


26/08/04 18:41:21 WARN DAGScheduler: Broadcasting large task binary with size 3.0 MiB


26/08/04 18:41:22 WARN DAGScheduler: Broadcasting large task binary with size 3.0 MiB


26/08/04 18:41:24 WARN DAGScheduler: Broadcasting large task binary with size 3.0 MiB


+-----+---+----+-----------------+-----+------------------+------------------+
|month|day|hour|       pickupZone|trips|      avg_distance|        avg_amount|
+-----+---+----+-----------------+-----+------------------+------------------+
|    1|  1|   0|    Alphabet City|  153|2.8382352941176476| 14.64058823529411|
|    1|  1|   0|          Astoria|  113|2.8107079646017707|13.155044247787597|
|    1|  1|   0|     Astoria Park|    1|              1.67|             11.16|
|    1|  1|   0|     Battery Park|    9|3.7866666666666666|16.183333333333334|
|    1|  1|   0|Battery Park City|  131|3.8383206106870236| 17.92351145038167|
+-----+---+----+-----------------+-----+------------------+------------------+
only showing top 5 rows



### Заполнение пропусков

Загрузим список зон и создадим набор данных с записями со всеми зонами на каждый час каждого дня с января по март.

In [59]:
val reference = spark
                  .read
                  .format("csv")
                  .option("header", "true")
                  .option("inferSchema", "true")
                  .load("data/taxi_zone_lookup.csv")
                  .select($"Borough".as("pickupBorough"), $"Zone".as("pickupZone"))
                  .withColumn("hour", explode(lit((0 to 23).toArray)))
                  .withColumn("day", explode(lit((1 to 31).toArray)))
                  .withColumn("month", explode(lit((1 to 3).toArray)))
                  .filter(!($"month" === 2 && $"day" > 29))
                  .select("month", "day", "hour", "pickupBorough", "pickupZone")
                  .orderBy("month", "day", "hour", "pickupBorough", "pickupZone")

reference: Dataset[Row] = [month: int, day: int ... 3 more fields]

In [60]:
reference.show(5)

+-----+---+----+-------------+--------------------+
|month|day|hour|pickupBorough|          pickupZone|
+-----+---+----+-------------+--------------------+
|    1|  1|   0|        Bronx|Allerton/Pelham G...|
|    1|  1|   0|        Bronx|        Bedford Park|
|    1|  1|   0|        Bronx|             Belmont|
|    1|  1|   0|        Bronx|          Bronx Park|
|    1|  1|   0|        Bronx|           Bronxdale|
+-----+---+----+-------------+--------------------+
only showing top 5 rows



In [61]:
val data = reference
             .join(
                tripsAgg,
                Seq("month", "day", "hour", "pickupZone"), "leftouter")
             .na
             .fill(0, Seq("trips", "avg_distance", "avg_amount"))

data: DataFrame = [month: int, day: int ... 6 more fields]

In [63]:
data.orderBy("month", "day", "hour", "pickupZone", "pickupBorough").show(5)

26/08/04 18:46:13 WARN DAGScheduler: Broadcasting large task binary with size 3.0 MiB


26/08/04 18:46:15 WARN DAGScheduler: Broadcasting large task binary with size 3.0 MiB


+-----+---+----+--------------------+-------------+-----+------------------+------------------+
|month|day|hour|          pickupZone|pickupBorough|trips|      avg_distance|        avg_amount|
+-----+---+----+--------------------+-------------+-----+------------------+------------------+
|    1|  1|   0|Allerton/Pelham G...|        Bronx|    0|               0.0|               0.0|
|    1|  1|   0|       Alphabet City|    Manhattan|  153|2.8382352941176476| 14.64058823529411|
|    1|  1|   0|       Arden Heights|Staten Island|    0|               0.0|               0.0|
|    1|  1|   0|Arrochar/Fort Wad...|Staten Island|    0|               0.0|               0.0|
|    1|  1|   0|             Astoria|       Queens|  113|2.8107079646017707|13.155044247787597|
+-----+---+----+--------------------+-------------+-----+------------------+------------------+
only showing top 5 rows

